# Tables gen (.tex)

In [1]:
import pandas as pd
import torch
from torch.utils.data import TensorDataset, ConcatDataset
from data_utils import extract_TensorDataset, extract_boundary
from load_store_utils import resume_model
from typing import List
import numpy as np

PDE = "AllenCahn"
ACTUAL_MODE = "PINN"
LR = "_CosLrAnnealing"
MEMORY = "LongMemory"
BD = "_LocalFullBd" # _LocalFullBd _LocalBd
LIMITED_MEMORY = True
N_EPOCHS = 200
DWA_MODE = "Norm1"
PARAM_INDEX = None
CLIP = "GradClip"

FONT = "Cambria !important"
FONT_SIZE = "12pt !important"

YELLOW = ["#FFF3D1", "#FFE7A5", "#FFD483", "#B36430"]
ORANGE = ["#FFE6B8", "#FFD8B8", "#FFBF8A", "#D15E00", "#A34900"]
PURPLE = ["#EDE6FF", "#C9BCEA", "#A48BDE", "#6500A3"]
BROWN = ["#E8CECE", "#D9AFAF", "#BC7171", "#8E4343"]
RED = ["#FFD0D0", "#D59999", "#D17878", "#8E4343"]
BLUE = ["#E6E9FF", "#B8C2FF", "#687FFF", "#4D5AD0"]#"#8799FF"
GRAY = ["#E5E5E5", "#968C8C", "#605C5C",]
# -----------------------------------
if PARAM_INDEX is None:
    DIR = "MultiTask"
else:
    DIR = f"Task{PARAM_INDEX}"

if LIMITED_MEMORY:
    MEM = "LimitedMemory"
    MEM_STR = "limited memory"
else:
    MEM = "UnlimitedMemory"
    MEM_STR = "unlimited memory"

if DWA_MODE == "Std":
    DWA_STR = "standard weighting"
elif DWA_MODE == "Norm1":
    DWA_STR = "sum-to-one weighting"
else:
    DWA_STR = "sum-to-K weighting"

In [2]:
def subsample(
    datasets: List[ConcatDataset],
    n_samples: int,
    seed: int = 42
    ) -> List[ConcatDataset]:
    
    reduced_datasets = []
    last_seed = seed
    for concat_ds in datasets:
        seeds = [last_seed+i for i in range(len(concat_ds))]
        last_seed = seeds[-1]
        reduced_concat_ds = []
        for ds, seed in zip(concat_ds.datasets, seeds):
            torch.manual_seed(seed)
            indices = torch.randperm(len(ds))
            indices = indices[:n_samples]
            reduced_cols = [col[indices] for col in ds.tensors]
            reduced_concat_ds.append(TensorDataset(*reduced_cols))
        reduced_datasets.append(ConcatDataset(reduced_concat_ds))
    return reduced_datasets

def merge_ds(datasets: List[TensorDataset]) -> TensorDataset:
    cols = None   
    for ds in datasets:
        new_cols = list(ds.tensors)
        if cols is None:
            cols = new_cols
        else:
            for i, col in enumerate(new_cols):
                cols[i] = torch.cat([cols[i], col])
    return TensorDataset(*cols)

def prepare_dataset(datasets: List[ConcatDataset]) -> ConcatDataset:
    n_snapshots = len(datasets[0].datasets)
    data = [None for _ in range(n_snapshots)]
    for i in range(n_snapshots):
        data[i] = merge_ds([concat_ds.datasets[i] for concat_ds in datasets])
    return ConcatDataset(data)

def make_bold_min(row):
    is_min = row == row.min()
    return["font-weight: bold !important; color: darkred !important;" if v else '' for v in is_min]

def to_scientific(val):
    if pd.isna(val) or isinstance(val, str):
        return val
    # Format to scientific notation with 2 decimal places
    return "{:.2e}".format(val)

def latex_sci(val):
    if pd.isna(val) or val == 0:
        return f"{val}"
    # Format to sci notation
    val = "{:.2e}".format(val)
    base, exponent = val.split("e")
    # Remove leading zeros and plus signs from exponent
    exponent = int(exponent) 
    return f"${base} \\times 10^{{{exponent}}}$"

def latex_bold_formatter(x, is_bold):
    if is_bold:
        return f"\\mathbf{{{latex_sci(x)}}}"
    return latex_sci(x)

def strip_math(s):
    return s[1:-1] if s.startswith("$") and s.endswith("$") else s

In [3]:
full_datasets = torch.load(f"{PDE}/data/full_datasets.pth", weights_only=False).datasets
unlabeled_datasets = torch.load(f"{PDE}/data/unlabeled_datasets.pth", weights_only=False).datasets
dev_datasets = torch.load(f"{PDE}/data/dev_datasets.pth", weights_only=False).datasets
train_datasets = torch.load(f"{PDE}/data/train_datasets.pth", weights_only=False).datasets
val_datasets = torch.load(f"{PDE}/data/val_datasets.pth", weights_only=False).datasets
inter_test_datasets = torch.load(f"{PDE}/data/inter_test_datasets.pth", weights_only=False).datasets
intra_test_datasets = torch.load(f"{PDE}/data/intra_test_datasets.pth", weights_only=False).datasets

n_params = 2 # number of xi_j

if PARAM_INDEX != None:
    train_dataset = [dev_datasets[PARAM_INDEX]]
    train_boundary_dataset = [ConcatDataset([extract_boundary(dev_datasets[PARAM_INDEX])])]
    unlabeled_dataset = [unlabeled_datasets[PARAM_INDEX]]
    intra_test_dataset = [intra_test_datasets[PARAM_INDEX]]
else:
    train_dataset = dev_datasets
    train_boundary_dataset = [ConcatDataset([extract_boundary(ds)]) for ds in dev_datasets]
    unlabeled_dataset = unlabeled_datasets[:len(dev_datasets)]
    intra_test_dataset = intra_test_datasets

train_data = prepare_dataset(train_dataset)
intra_test_data = prepare_dataset(intra_test_dataset).datasets[0]
inter_test_data = prepare_dataset(inter_test_datasets).datasets[0]

# Global performances

## Full

In [4]:
model_dir = f"{PDE}/FullDomainLearning/{DIR}/{ACTUAL_MODE}_{DWA_MODE}_{CLIP}_{N_EPOCHS}/models2/trial0"
caption_prefix = f"Full domain learning, Residual+BCs, {DWA_STR}, "
label_prefix = f"Full{DWA_MODE}DomainLearning"
colors = {
    "index": RED[3][1:], 
    "columns": RED[2][1:],
    "highlight": RED[0][1:]
}
step = 3
datasets = [(intra_test_data, "Intra Test"), (inter_test_data, "Inter Test")]
row_ids = ["Out Loss", "Der1 Loss", "Der2 Loss", "Res Loss"]

# ---------------------------------------------
rows = [[] for _ in row_ids]
for dataset, dataset_name in datasets:
    model = resume_model(model_path=f"{model_dir}/model.pth")
    out_loss, der_loss, hes_loss, res_loss = \
        model.evaluate(dataset=dataset)
    rows[0].append(out_loss.item())
    rows[1].append(der_loss.item())
    rows[2].append(hes_loss.item())
    rows[3].append(res_loss.item())

df = pd.DataFrame(
    data=rows,
    columns=[dataset_name for _, dataset_name in datasets],
    index=row_ids
)

df.index = [f"\\color[HTML]{{{colors['index']}}}{{{x}}}" for x in df.index]
df.columns = [f"\\color[HTML]{{{colors['columns']}}}\\textbf{{{x}}}" for x in df.columns]
        
is_min = df.eq(df.min(axis=1), axis=0)
df_latex = df.copy()
for i in df.index:
    for j in df.columns:
        val = df.loc[i, j]
        if is_min.loc[i, j]:
            df_latex.loc[i, j] = (
                f"${{{strip_math(latex_sci(val))}}}$"
            )
        else:
            df_latex.loc[i, j] = latex_sci(val)
            
latex_tabular = df_latex.to_latex(
    column_format="l" + "c" * len(df.columns),
    escape=False
)
caption = f"{caption_prefix}global performances."
label = f"{label_prefix}GlobalPerformances"
latex_code = f"""
\\begin{{table}}[H]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
{latex_tabular}\\end{{table}}
"""
with open(
    f"{PDE}/FullDomainLearning/{DIR}/Table{N_EPOCHS}.tex",
    "w"
) as f:
    f.write(latex_code)

/tmp/ipykernel_9963/3339985987.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '${3.67 \times 10^{-6}}$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[i, j] = (
/tmp/ipykernel_9963/3339985987.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '$8.22 \times 10^{-4}$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[i, j] = latex_sci(val)


## Distill

In [5]:
action = "Distill"
starting_modes = ["FineTune"]
caption_prefixes = [ f"Distillation, {MEM_STR}, {DWA_STR}, fine tuning, "]
label_prefixes = [f"Distill{DWA_MODE}{MEM}FineTuning"]
colors = {
    "index": ORANGE[4][1:], 
    "columns": ORANGE[3][1:],
    "highlight": ORANGE[0][1:]
}
columns = ["Out", "Der1", "Der2", "Out+Der1", "Out+Der1+Der2"]
step = 3
datasets = [(intra_test_data, "Intra Test"), (inter_test_data, "Inter Test")]
action_modes = ["Output", "Derivative", "Hessian", "Output+Derivative", "Output+Derivative+Hessian"]
row_ids = ["Out Loss", "Der1 Loss", "Der2 Loss", "Res Loss"]

# ---------------------------------------------

for dataset, dataset_name in datasets:
    for starting_mode, caption_prefix, label_prefix in zip(starting_modes, caption_prefixes, label_prefixes):
        rows = [[] for _ in row_ids]
        for action_mode in action_modes:    
            model_dir = f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/{action}/{action_mode}/{starting_mode}{MEMORY}{N_EPOCHS}/Task{step}/models/trial0"
            model = resume_model(model_path=f"{model_dir}/model.pth")
            out_loss, der_loss, hes_loss, res_loss = \
                model.evaluate(dataset=dataset)
            rows[0].append(out_loss.item())
            rows[1].append(der_loss.item())
            rows[2].append(hes_loss.item())
            rows[3].append(res_loss.item())

        df = pd.DataFrame(
            data=rows,
            columns=columns,
            index=row_ids
        )

        df.index = [f"\\color[HTML]{{{colors['index']}}}{{{x}}}" for x in df.index]
        df.columns = [f"\\color[HTML]{{{colors['columns']}}}\\textbf{{{x}}}" for x in df.columns]

        is_min = df.eq(df.min(axis=1), axis=0)
        df_latex = df.copy()

        for i in df.index:
            for j in df.columns:
                val = df.loc[i, j]

                if is_min.loc[i, j]:
                    df_latex.loc[i, j] = (
                        f"\\cellcolor[HTML]{{{colors['highlight']}}}$\\mathbf{{{strip_math(latex_sci(val))}}}$"
                    )
                else:
                    df_latex.loc[i, j] = latex_sci(val)
        
        latex_tabular = df_latex.to_latex(
            column_format="l" + "c" * len(df.columns),
            escape=False
        )
        caption = f"{caption_prefix}{dataset_name.lower()} global performances."
        label = f"{label_prefix}{dataset_name.replace(' ', '')}GlobalPerformances"
        latex_code = f"""
\\begin{{table}}[H]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
{latex_tabular}\\end{{table}}
            """
        with open(
            f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/{action}/{dataset_name.replace(' ', '')}{starting_mode}Table{N_EPOCHS}.tex",
            "w"
        ) as f:
            f.write(latex_code)

/tmp/ipykernel_9963/3860471862.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '$6.36 \times 10^{-5}$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[i, j] = latex_sci(val)
/tmp/ipykernel_9963/3860471862.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '$1.72 \times 10^{-4}$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[i, j] = latex_sci(val)
/tmp/ipykernel_9963/3860471862.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '$7.35 \times 10^{-4}$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[i, j] = latex_sci(val)
/tmp/ipykernel_9963/3860471862.py:52: FutureWarni

## EWC

In [6]:
if LIMITED_MEMORY:
    action = "EWC"
    starting_modes = ["FineTune"]
    warmup_modes = ["Warmup1"]
    caption_prefixes = [f"EWC, {DWA_STR}, {MEM_STR}, warmup 1, "]
    label_prefixes = [f"EWC{DWA_MODE}{MEM}Warmup1"]
    colors = {
        "index": PURPLE[3][1:], 
        "columns": PURPLE[2][1:],
        "highlight": PURPLE[0][1:]
    }
    columns = ["Fine Tuning"]#["Warmup 1", "Warmup 5"]
    step = 3
    datasets = [(intra_test_data, "Intra Test"), (inter_test_data, "Inter Test")]
    action_modes = ["alpha_1.0/weight_auto/warmup1_decay1.0"]
    row_ids = ["Out Loss", "Der1 Loss", "Der2 Loss", "Res Loss"]
    
    # ---------------------------------------------
    
    for dataset, dataset_name in datasets:
        for action_mode, warmup_mode, caption_prefix, label_prefix in zip(action_modes, warmup_modes, caption_prefixes, label_prefixes):
            rows = [[] for _ in row_ids]
            for starting_mode in starting_modes:
                model_dir = f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/{action}/{action_mode}/{starting_mode}{MEMORY}{N_EPOCHS}/Task{step}/models/trial0"
                model = resume_model(model_path=f"{model_dir}/model.pth")
                out_loss, der_loss, hes_loss, res_loss = \
                    model.evaluate(dataset=dataset)
                rows[0].append(out_loss.item())
                rows[1].append(der_loss.item())
                rows[2].append(hes_loss.item())
                rows[3].append(res_loss.item())
    
            df = pd.DataFrame(
                data=rows,
                columns=columns,
                index=row_ids
            )
    
            df.index = [f"\\color[HTML]{{{colors['index']}}}{{{x}}}" for x in df.index]
            df.columns = [f"\\color[HTML]{{{colors['columns']}}}\\textbf{{{x}}}" for x in df.columns]
            
            is_min = df.eq(df.min(axis=1), axis=0)
            df_latex = df.copy()
    
            for i in df.index:
                for j in df.columns:
                    val = df.loc[i, j]
    
                    if is_min.loc[i, j]:
                        df_latex.loc[i, j] = (
                            f"\\cellcolor[HTML]{{{colors['highlight']}}}$\\mathbf{{{strip_math(latex_sci(val))}}}$"
                        )
                    else:
                        df_latex.loc[i, j] = latex_sci(val)
            
            latex_tabular = df_latex.to_latex(
                column_format="l" + "c" * len(df.columns),
                escape=False
            )
            caption = f"{caption_prefix}{dataset_name.lower()} global performances."
            label = f"{label_prefix}{dataset_name.replace(' ', '')}GlobalPerformances"
            latex_code = f"""
\\begin{{table}}[H]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
{latex_tabular}\\end{{table}}
                """
            with open(
                f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/{action}/{dataset_name.replace(' ', '')}{warmup_mode}Table{N_EPOCHS}.tex",
                "w"
            ) as f:
                f.write(latex_code)

/tmp/ipykernel_9963/3881011802.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '\cellcolor[HTML]{EDE6FF}$\mathbf{1.34 \times 10^{-3}}$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[i, j] = (
/tmp/ipykernel_9963/3881011802.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '\cellcolor[HTML]{EDE6FF}$\mathbf{7.65 \times 10^{-3}}$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[i, j] = (


## Replay

In [7]:
action = "Replay"
starting_modes = ["FineTune"]
caption_prefix = f"Replay Residual+BCs, {DWA_STR}, {MEM_STR}, "
label_prefix = f"Replay{DWA_MODE}{MEM}"
colors = {
    "index": BLUE[3][1:], 
    "columns": BLUE[2][1:],
    "highlight": BLUE[0][1:]
}
step = 3
datasets = [(intra_test_data, "Intra Test"), (inter_test_data, "Inter Test")]
columns = ["Fine Tuning"]
action_mode = "Residual+Boundary"
row_ids = ["Out Loss", "Der1 Loss", "Der2 Loss", "Res Loss"]

# ---------------------------------------------

for dataset, dataset_name in datasets:
    rows = [[] for _ in row_ids]
    for starting_mode in starting_modes:
        model_dir = f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/{action}/{action_mode}/{starting_mode}{MEMORY}{N_EPOCHS}/Task{step}/models/trial0"
        model = resume_model(model_path=f"{model_dir}/model.pth")
        out_loss, der_loss, hes_loss, res_loss = \
            model.evaluate(dataset=dataset)
        rows[0].append(out_loss.item())
        rows[1].append(der_loss.item())
        rows[2].append(hes_loss.item())
        rows[3].append(res_loss.item())

    df = pd.DataFrame(
        data=rows,
        columns=columns,
        index=row_ids
    )

    df.index = [f"\\color[HTML]{{{colors['index']}}}{{{x}}}" for x in df.index]
    df.columns = [f"\\color[HTML]{{{colors['columns']}}}\\textbf{{{x}}}" for x in df.columns]
    
    is_min = df.eq(df.min(axis=1), axis=0)
    df_latex = df.copy()
    for i in df.index:
        for j in df.columns:
            val = df.loc[i, j]
            if is_min.loc[i, j]:
                df_latex.loc[i, j] = (
                    f"\\cellcolor[HTML]{{{colors['highlight']}}}$\\mathbf{{{strip_math(latex_sci(val))}}}$"
                )
            else:
                df_latex.loc[i, j] = latex_sci(val)

    latex_tabular = df_latex.to_latex(
        column_format="l" + "c" * len(df.columns),
        escape=False
    )
    caption = f"{caption_prefix}{dataset_name.lower()} global performances."
    label = f"{label_prefix}{dataset_name.replace(' ', '')}GlobalPerformances"
    latex_code = f"""
\\begin{{table}}[H]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
{latex_tabular}\\end{{table}}
    """
    
    with open(
        f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/{action}/{dataset_name.replace(' ', '')}Table{N_EPOCHS}.tex",
        "w"
    ) as f:
        f.write(latex_code)

/tmp/ipykernel_9963/622669640.py:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '\cellcolor[HTML]{E6E9FF}$\mathbf{2.56 \times 10^{-4}}$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[i, j] = (
/tmp/ipykernel_9963/622669640.py:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '\cellcolor[HTML]{E6E9FF}$\mathbf{2.38 \times 10^{-3}}$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[i, j] = (


## Forget

In [8]:
action = "Forget"
starting_modes = ["FineTune"]
caption_prefix = f"Forget, {DWA_STR}, "
label_prefix = f"Forget{DWA_MODE}"

colors = {
    "index": GRAY[2][1:], 
    "columns": GRAY[1][1:],
    "highlight": GRAY[0][1:]
}

step = 3
datasets = [(intra_test_data, "Intra Test"), (inter_test_data, "Inter Test")]
columns = ["Fine Tuning"]
row_ids = ["Out Loss", "Der1 Loss", "Der2 Loss", "Res Loss"]

# ---------------------------------------------

latex_tabular = ""
for dataset, dataset_name in datasets:
    rows = [[] for _ in row_ids]
    for starting_mode in starting_modes:
        f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{action}/{dataset_name.replace(' ', '')}{starting_mode}Table{N_EPOCHS}.tex",
        
        model_dir = f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{action}/{starting_mode}{N_EPOCHS}/Task{step}/models/trial0"
        model = resume_model(model_path=f"{model_dir}/model.pth")
        out_loss, der_loss, hes_loss, res_loss = \
            model.evaluate(dataset=dataset)
        rows[0].append(out_loss.item())
        rows[1].append(der_loss.item())
        rows[2].append(hes_loss.item())
        rows[3].append(res_loss.item())

    df = pd.DataFrame(
        data=rows,
        columns=columns,
        index=row_ids
    )

    df.index = [f"\\color[HTML]{{{colors['index']}}}{{{x}}}" for x in df.index]
    df.columns = [f"\\color[HTML]{{{colors['columns']}}}\\textbf{{{x}}}" for x, starting_mode in zip(df.columns, starting_modes)]
    
    is_min = df.eq(df.min(axis=1), axis=0)
    df_latex = df.copy()
    for i in df.index:
        for j in df.columns:
            val = df.loc[i, j]
            if is_min.loc[i, j]:
                df_latex.loc[i, j] = (
                    f"\\cellcolor[HTML]{{{colors['highlight']}}}$\\mathbf{{{strip_math(latex_sci(val))}}}$"
                )
            else:
                df_latex.loc[i, j] = latex_sci(val)

    latex_tabular = df_latex.to_latex(
        column_format="l" + "c" * len(df.columns),
        escape=False
    )
    caption = f"{caption_prefix}{dataset_name.lower()} global performances."
    label = f"{label_prefix}{dataset_name.replace(' ', '')}GlobalPerformances"
    latex_code = f"""
\\begin{{table}}[H]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
{latex_tabular}\\end{{table}}
    """
print(latex_code)        
    #with open(
    #    f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{action}/{dataset_name.replace(' ', '')}Table{N_EPOCHS}.tex",
    #    "w"
    #) as f:
    #    f.write(latex_code)

/tmp/ipykernel_9963/3254108173.py:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '\cellcolor[HTML]{E5E5E5}$\mathbf{3.49 \times 10^{-3}}$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[i, j] = (



\begin{table}[H]
\centering
\caption{Forget, sum-to-one weighting, inter test global performances.}
\label{ForgetNorm1InterTestGlobalPerformances}
\begin{tabular}{lc}
\toprule
 & \color[HTML]{968C8C}\textbf{Fine Tuning} \\
\midrule
\color[HTML]{605C5C}{Out Loss} & \cellcolor[HTML]{E5E5E5}$\mathbf{1.17 \times 10^{-2}}$ \\
\color[HTML]{605C5C}{Der1 Loss} & \cellcolor[HTML]{E5E5E5}$\mathbf{1.49 \times 10^{-1}}$ \\
\color[HTML]{605C5C}{Der2 Loss} & \cellcolor[HTML]{E5E5E5}$\mathbf{4.09 \times 10^{0}}$ \\
\color[HTML]{605C5C}{Res Loss} & \cellcolor[HTML]{E5E5E5}$\mathbf{1.52 \times 10^{-2}}$ \\
\bottomrule
\end{tabular}
\end{table}
    


/tmp/ipykernel_9963/3254108173.py:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '\cellcolor[HTML]{E5E5E5}$\mathbf{1.17 \times 10^{-2}}$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[i, j] = (


In [9]:
colors = [{
    "index": ORANGE[4][1:], 
    "columns": ORANGE[3][1:],
    "highlight": ORANGE[0][1:]
},
{
    "index": BLUE[3][1:], 
    "columns": BLUE[2][1:],
    "highlight": BLUE[0][1:]
}]
color_scales = {
    "orange": ["FFB780", "FFBB56", "FFD981", "FFF2BE"],
    "blue":   ["99B1FF", "B2C3FD", "D4DEFF", "ECF5FF"],
    "purple": ["9932CC", "BA55D3", "D8BFD8", "E6E6FA", "F8F8FF"]
}

# Choose which one you want to use for this specific table
gradients = [color_scales["orange"], color_scales["blue"]]

full_model = f"{PDE}/FullDomainLearning/{DIR}/{ACTUAL_MODE}_{DWA_MODE}_{CLIP}_{N_EPOCHS}/models2/trial0/model.pth"
forget_models = {
    "FineTune": f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/Forget/FineTune{N_EPOCHS}/Task3/models/trial0/model.pth"
}
ewc_models = {
    "FineTune": f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/EWC/alpha_1.0/weight_auto/warmup1_decay1.0/FineTuneLongMemory{N_EPOCHS}/Task3/models/trial0/model.pth"
}
replay_models = {
    "FineTune": f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/Replay/Residual+Boundary/FineTune{MEMORY}{N_EPOCHS}/Task3/models/trial0/model.pth"
}
distill_modes = ["Output", "Derivative", "Hessian", "Output+Derivative", "Output+Derivative+Hessian"]
distill_models = {}
for mode in distill_modes:
    distill_models[mode] = {
        "FineTune": f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/Distill/{mode}/FineTune{MEMORY}{N_EPOCHS}/Task3/models/trial0/model.pth"
    }

distill_modes2 = ["Out", "Der1", "Der2", "Out+Der1", "Out+Der1+Der2"]
distill_ids = []
distill_models2 = []
for mode, mode2 in zip(distill_modes, distill_modes2):
    distill_ids.append(f"Distill {mode2} FT")
    distill_models2.append(distill_models[mode]["FineTune"])
modes = ["FT"]
columns = ["Out Loss", "Der1 Loss", "Der2 Loss", "Res Loss"]
row_ids = \
    ["Full Learning"] + \
    [f"Forget {s}" for s in modes] + \
    [f"EWC {s}" for s in modes] + \
    [f"Replay Res+BC {s}" for s in modes] + \
    distill_ids
modes = ["FineTune"]
models = \
    [full_model] + \
    [forget_models[s] for s in modes] + \
    [ewc_models[s] for s in modes] + \
    [replay_models[s] for s in modes] + \
    distill_models2

datasets = [(colors[0], gradients[0], intra_test_data, "Intra Test"), (colors[1], gradients[1], inter_test_data, "Inter Test")]

for color, gradient, ds, ds_name in datasets:
    rows = []
    for model_file in models:
        model = resume_model(model_path=model_file)
        rows.append(model.evaluate(ds))
    
    df = pd.DataFrame(
        data=rows,
        columns=columns,
        index=row_ids
    )

    df.index = [f"\\color[HTML]{{{color['index']}}}{{{x}}}" for x in df.index]
    df.columns = [f"\\color[HTML]{{{color['columns']}}}\\textbf{{{x}}}" for x in df.columns]
    
    ranks = df.rank(axis=0, method='min').astype(int)

    df_latex = df.copy()
    for i in df.index:
        for j in df.columns:
            val = df.loc[i, j]
            r = ranks.loc[i, j]

            if r <= len(gradient):
                hex_color = gradient[r - 1]
                formatted_val = strip_math(latex_sci(val))
                if r == 1:
                    formatted_val = f"\\mathbf{{{formatted_val}}}"
                df_latex.loc[i, j] = f"\\cellcolor[HTML]{{{hex_color}}}${formatted_val}$"
            else:
                df_latex.loc[i, j] = latex_sci(val)

        latex_tabular = df_latex.to_latex(
            column_format="l" + "c" * len(df.columns),
            escape=False
        )
        caption = f"{ds_name.lower()} global performances."
        label = f"{ds_name.replace(' ', '')}GlobalPerformances"
        latex_code = f"""
\\begin{{table}}[H]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
{latex_tabular}\\end{{table}}
        """
print(latex_code)
    #with open(
    #    f"{PDE}/{ds_name.replace(' ', '')}TISummaryTable{N_EPOCHS}.tex",
    #    "w"
    #) as f:
    #    f.write(latex_code)


\begin{table}[H]
\centering
\caption{inter test global performances.}
\label{InterTestGlobalPerformances}
\begin{tabular}{lcccc}
\toprule
 & \color[HTML]{687FFF}\textbf{Out Loss} & \color[HTML]{687FFF}\textbf{Der1 Loss} & \color[HTML]{687FFF}\textbf{Der2 Loss} & \color[HTML]{687FFF}\textbf{Res Loss} \\
\midrule
\color[HTML]{4D5AD0}{Full Learning} & \cellcolor[HTML]{99B1FF}$\mathbf{8.22 \times 10^{-4}}$ & \cellcolor[HTML]{99B1FF}$\mathbf{1.05 \times 10^{-2}}$ & \cellcolor[HTML]{99B1FF}$\mathbf{4.32 \times 10^{-1}}$ & \cellcolor[HTML]{99B1FF}$\mathbf{9.11 \times 10^{-4}}$ \\
\color[HTML]{4D5AD0}{Forget FT} & $1.17 \times 10^{-2}$ & $1.49 \times 10^{-1}$ & $4.09 \times 10^{0}$ & $1.52 \times 10^{-2}$ \\
\color[HTML]{4D5AD0}{EWC FT} & $7.65 \times 10^{-3}$ & $8.11 \times 10^{-2}$ & $2.74 \times 10^{0}$ & $9.97 \times 10^{-3}$ \\
\color[HTML]{4D5AD0}{Replay Res+BC FT} & $2.38 \times 10^{-3}$ & $3.81 \times 10^{-2}$ & $2.60 \times 10^{0}$ & $3.56 \times 10^{-3}$ \\
\color[HTML]{4D5AD0}{Dist